In [ ]:
import os, sys, torch, random, glob
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm
# Aseguramos que Python encuentra las librerías desde cualquier carpeta
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from NoisyUAV.funciones.cargador import cargar_muestra
from NoisyUAV.funciones.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico
from NoisyUAV.modelos.burst_cvcnn import BurstCVCNN

In [ ]:
# 1. Buscar archivos Target 2 a 10dB (Alta SNR para ver el comportamiento óptimo)
archivos = glob.glob(r"C:\TFM_data\NoisyUAV\drone_RF_data\*target0_snr10.pt")
if not archivos:
    print("❌ No se encontraron archivos.")
else:
    # IMPORTANTE: 3 muestras máximo por lote para no saturar la RAM del navegador
    muestras_azar = random.sample(archivos, min(5, len(archivos)))
    # 2. Configuración CFAR RELAJADA (Queremos extraer todo para luego filtrar dinámicamente)
    FS = 14e6
    NPERSEG = 2048
    Z_THRESH_RELAXED = 2.0
    MIN_Z_ABS_RELAXED = 2.0
    Z_THRESH_HARD = 3.5
    MIN_Z_ABS_HARD = 4.0
    print("======================================================================")
    print("  SIMULADOR CURRICULUM: UMBRAL DINÁMICO (Z_MAX - 10%)")
    print("======================================================================")
    for ruta in muestras_azar:
        filename = os.path.basename(ruta)
        iq, _, _, _ = cargar_muestra(ruta)
        
        t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
            iq, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH_HARD, 
            min_burst_ms=0.25, merge_gap_ms=1.0, min_z_abs=MIN_Z_ABS_HARD,
            bg_mult=4.0, max_bins_frac=0.17, smooth_ms=0.2, adaptive_window_ms=10
        )
        
        print(f"\n📁 {filename} | {len(bursts)} bursts detectados:")
        if len(bursts) == 0:
            print("  (Ninguno, la señal es puro ruido blanco para el CFAR)")
            continue
            
        # ---------------------------------------------------------
        # LÓGICA DE UMBRAL DINÁMICO (LA IDEA DEL USUARIO)
        # ---------------------------------------------------------
        max_z_abs = max(abs(b['z_peak']) for b in bursts)
        # Umbral dinámico con 10% de holgura
        umbral_dinamico = max_z_abs * 0.75  
        
        print(f"  --> Z_PEAK MÁXIMO DE LA SALA: {max_z_abs:.1f}")
        print(f"  --> UMBRAL DE CORTE (-10%):   {umbral_dinamico:.1f}")
        
        for i, b in enumerate(bursts):
            z_abs = abs(b['z_peak'])
            accion = "✅ GUARDAR" if z_abs >= umbral_dinamico else "❌ TIRAR A LA BASURA"
            print(f"  [B{i+1:02d}] dur={b['dur_ms']:4.1f}ms | z_peak={z_abs:5.1f} | {accion}")
            
        # 4. PLOTEAR PARA VALIDAR VISUALMENTE
        fig = plot_muestra(
            iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
            fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH_RELAXED,
            bg_mult=4.0, max_bins_frac=0.25, adaptive_window_ms=10,
            titulo=f"Filtro Dinámico (-10%): {filename}"
        )
        fig.show()

In [ ]:
# 1. Cargar el dataset de entrenamiento del Teacher
csv_path = r"c:\repos\DroneDetectionRF\NoisyUAV\curriculum_teacher_v1\teacher_dataset_high_snr.csv"
df_teacher = pd.read_csv(csv_path)
df_teacher = df_teacher[df_teacher['is_dummy'] == False]

# 2. Elegir el Dron (Target) que queremos investigar
TARGET_A_INVESTIGAR = 6  # <-- PRUEBA CON 0, 2, o 6 para ver la tragedia
df_target = df_teacher[df_teacher['target_multiclass'] == TARGET_A_INVESTIGAR]

print(f"Hay {len(df_target)} bursts de Target {TARGET_A_INVESTIGAR} en el dataset.")

# 3. Coger un burst aleatorio
row = df_target.sample(1).iloc[0]
ruta_archivo = os.path.join(r"C:\TFM_data\NoisyUAV\drone_RF_data", row['file_path'])

print("==================================================")
print(f"Analizando Archivo Usado en Entrenamiento: {row['file_path']}")
print(f"Datos del CSV -> t_start: {row['t_start']:.2f} ms | t_end: {row['t_end']:.2f} ms | dur: {row['dur_ms']:.2f} ms")
print("==================================================")

# 4. Cargar la muestra
from NoisyUAV.funciones.cargador import cargar_muestra
iq_tensor, _, original_target, original_snr = cargar_muestra(ruta_archivo)

# 5. Volver a pasar el detector para pintar el fondo
from NoisyUAV.funciones.detector_entropia import detectar_bursts, plot_muestra
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, _ = detectar_bursts(
    iq_tensor, fs=14e6, nperseg=2048, z_thresh=2.0,
    min_burst_ms=0.25, merge_gap_ms=1.0,
    bg_mult=4.0, max_bins_frac=0.25
)

# 6. Crear un burst artificial SÓLO con los datos del CSV para que se resalte en la gráfica
burst_del_csv = [{
    't0': row['t_start'],
    't1': row['t_end'],
    'dur_ms': row['dur_ms'],
    'z_peak': row['z_peak'],
    'drop_b': row['drop_b'],
    'n_act': row['n_act_burst'],
    'i0': max(0, int(row['t_start'] / (t_ms[1]-t_ms[0]))),
    'i1': min(len(t_ms)-1, int(row['t_end'] / (t_ms[1]-t_ms[0])))
}]

# 7. Dibujar
fig_2d = plot_muestra(
    iq_tensor, t_ms=t_ms, H=H, H_smooth=H_smooth, umbral_v=umbral_v, nf_v=nf_v, ns=ns,
    n_active=n_active, bursts=burst_del_csv, fs=14e6, nperseg=2048,
    titulo=f"LO QUE VIO LA RED AL ENTRENAR (Target {TARGET_A_INVESTIGAR} - SNR {original_snr})"
)
fig_2d.show()
